# 04 · Our contribution: fine-tuning Whisper-small and wav2vec 2.0 XLS-R on our train split

Runs on Kaggle with two GPUs. **Settings → Accelerator: GPU T4 x2, Internet: On.** No manual uploads.
`SYSTEMS` lists the systems of one Kaggle session, at most one per GPU (we ran both together).

`scripts/finetune.py` trains on the train split, chooses the checkpoint and the decoding settings
on dev, and decodes test once at the end (settings: `configs/finetune.yaml`), in a virtual
environment with the pinned packages of `requirements/finetune.txt`. Each system runs in its own
process on its own GPU; the audio is decoded once before.
`TRIAL = True` runs 30 steps on 200 train clips and 20 dev/test clips into `/kaggle/working/trial`.

Output: `/kaggle/working/results_<system>.zip` (predictions, `.meta.json`, training log) and the
fine-tuned weights `/kaggle/working/<system>_best/` (not in the repository).

In [ ]:
REPO_URL = "https://github.com/eneserman/NLP.git"  # the public repository
BRANCH = "main"
SYSTEMS = ["whisper_small_ft", "xlsr_300m_ft"]
TRIAL = False

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception:
    print("No HF_TOKEN secret; using anonymous access")

In [ ]:
# The repository and a virtual environment with only our pinned packages live in /tmp, so that the
# preinstalled packages of the Kaggle image cannot conflict and /kaggle/working holds only the results.
import os
os.environ.pop("PYTHONPATH", None)          # nothing from the Kaggle image on the path of our environment
%cd /tmp
!rm -rf repo venv && git clone -q -b $BRANCH $REPO_URL repo
%cd /tmp/repo
!git log -1 --format='commit %H'
!pip install -q virtualenv==21.15.9 && python -m virtualenv -q /tmp/venv
PY = "/tmp/venv/bin/python"
!{PY} -m pip install -q -r requirements/finetune.txt
!{PY} -m pip freeze > /kaggle/working/pip_freeze.txt && cat /kaggle/working/pip_freeze.txt
!{PY} -c "import sys; print('sys.path:', sys.path)"
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "no GPU"

In [ ]:
!{PY} -m pytest -q tests/test_finetune.py tests/test_inference.py

In [ ]:
!{PY} scripts/unpack_audio.py

In [ ]:
import subprocess, threading
extra = ["--max-steps", "30", "--limit-train", "200", "--limit-eval", "20", "--out-dir", "/kaggle/working/trial"] if TRIAL else []
# 1. caches shared by all systems (features / decoded audio), built once
r = subprocess.run([PY, "scripts/finetune.py", "--system", SYSTEMS[0], "--prepare-only", *extra])
assert r.returncode == 0, "preparing the data failed"
n_gpu = int(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.count("GPU "))
assert len(SYSTEMS) <= n_gpu, f"{len(SYSTEMS)} systems but {n_gpu} GPUs"
# 2. one process per system, each on its own GPU
def stream(name, proc):
    for line in proc.stdout:
        print(f"[{name}] {line}", end="", flush=True)
procs = []
for gpu, s in enumerate(SYSTEMS):
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
    p = subprocess.Popen([PY, "scripts/finetune.py", "--system", s, *extra], env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    t = threading.Thread(target=stream, args=(s, p)); t.start()
    procs.append((s, p, t))
for s, p, t in procs:
    p.wait(); t.join()
    print("exit code", s, p.returncode, flush=True)

In [ ]:
for s in SYSTEMS:
    if TRIAL:
        !head -c 3000 /kaggle/working/trial/train_logs/{s}.json
        !head -n 3 /kaggle/working/trial/predictions/{s}.dev.jsonl
    else:
        !cp -r work/models/{s}/best /kaggle/working/{s}_best
        !zip -q -r /kaggle/working/results_{s}.zip results/predictions/{s}.* results/train_logs/{s}.json
!ls -lh /kaggle/working